In [ ]:
!pip install flask flask-cors transformers torch -q

# Download the official cloudflared binary (Cloudflare's tunnel client)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("cloudflared installed ✓")

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from transformers import pipeline
import threading, subprocess, re, time

app = Flask(__name__)
CORS(app)

print("Loading model...")
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print("Model ready!")

@app.route("/analyze", methods=["POST"])
def analyze():
    data = request.json
    text = data.get("text", "")
    if not text or len(text) < 5:
        return jsonify({"error": "Text too short"}), 400
    result = classifier(text[:512])[0]
    return jsonify({
        "label": result["label"],
        "score": round(result["score"] * 100, 1)
    })

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "tunnel": "cloudflare"})

def run_flask():
    app.run(port=5000, use_reloader=False)

t = threading.Thread(target=run_flask)
t.daemon = True
t.start()
time.sleep(1)
print("Flask running on port 5000 ✓")

In [ ]:
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait and extract the public URL from cloudflared's output
public_url = None
for line in proc.stdout:
    print(line, end="")
    match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print(f"\n{'='*50}")
        print(f"✅ Your Cloudflare URL: {public_url}")
        print(f"   Health check:        {public_url}/health")
        print(f"{'='*50}\n")
        break

In [ ]:
import requests

tests = [
    "This is absolutely amazing, I love it!",
    "Terrible experience, completely disappointed.",
    "It was okay, nothing special."
]

for text in tests:
    r = requests.post("http://localhost:5000/analyze", json={"text": text})
    d = r.json()
    emoji = "😊" if d["label"] == "POSITIVE" else "😞"
    print(f"{emoji} {d['label']} ({d['score']}%)  →  {text}")